# 🔥 Project IRIS: Heretic 7B Model Decensoring Pipeline

This Google Colab notebook runs [**Heretic**](https://github.com/p-e-w/heretic) to automatically abliterate refusal / censorship directions from **`Qwen/Qwen2.5-Coder-7B-Instruct`** and exports the resulting uncensored weights into **GGUF format (`Q4_K_M`)** for deployment in **Ollama** and **Project IRIS**.

### ⚠️ Important: Hardware Acceleration Setup
Before running, ensure hardware acceleration is enabled:
1. Go to **Runtime → Change runtime type**.
2. Select **T4 GPU** (free tier) or **A100 / L4** (Colab Pro).
3. Click **Save**.

### 1. Verify GPU Availability
Check that a CUDA-compatible GPU is attached.

In [ ]:
!nvidia-smi

### 2. Install Heretic & Acceleration Dependencies
Install `heretic-llm`, PyTorch, bitsandbytes (for 4-bit quantization during optimization), and Optuna.

In [ ]:
# Install heretic and core libraries
!pip install -U heretic-llm bitsandbytes accelerate optuna transformers huggingface_hub peft

# Ensure llama.cpp requirements for GGUF conversion are ready
!pip install gguf sentencepiece

### 3. Configure Model (7B) & Optimization Parameters
Using **Qwen/Qwen2.5-Coder-7B-Instruct** for superior reasoning, function calling, and zero refusals.

In [ ]:
import os

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"   # 7B Target HF model
OUTPUT_DIR = "/content/heretic_7b_output"     # Destination for abliterated weights
N_TRIALS = 35                                 # 35 trials is fast and effective for 7B on T4
QUANTIZATION = "bnb_4bit"                     # 4-bit takes ~5.5GB VRAM (fits easily in 16GB T4)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Target Model: {MODEL_ID}")
print(f"Optimization Trials: {N_TRIALS}")
print(f"Quantization: {QUANTIZATION}")

### 4. Run Heretic Abliteration & Optimization
Heretic will:
1. Benchmark the GPU for optimal batch size.
2. Extract harmless vs harmful activation residuals.
3. Run Optuna TPE to find the best abliteration kernel and LoRA weights.
4. Merge and export the decensored 7B model weights into `OUTPUT_DIR`.

In [ ]:
!heretic \
    --model "{MODEL_ID}" \
    --quantization "{QUANTIZATION}" \
    --n-trials {N_TRIALS} \
    --export-strategy merge \
    --export-path "{OUTPUT_DIR}"

### 5. Convert Abliterated 7B Weights to GGUF Format (`Q4_K_M`)
We clone `llama.cpp` and quantize the 7B model to `Q4_K_M` (~4.4 GB), allowing it to fit into RAM.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
%cd /content/llama.cpp
!make -j llama-quantize || cmake -B build && cmake --build build --config Release -t llama-quantize
!pip install -r requirements.txt

# Step A: Convert to f16 intermediate GGUF
GGUF_F16 = "/content/iris-qwen2.5-coder-7b-heretic.f16.gguf"
!python3 convert_hf_to_gguf.py "{OUTPUT_DIR}" --outfile "{GGUF_F16}" --outtype f16

# Step B: Quantize to Q4_K_M for optimal 8GB RAM local performance (~4.4 GB)
GGUF_FINAL = "/content/iris-qwen2.5-coder-7b-heretic.Q4_K_M.gguf"
if os.path.exists("./build/bin/llama-quantize"):
    !./build/bin/llama-quantize "{GGUF_F16}" "{GGUF_FINAL}" Q4_K_M
elif os.path.exists("./llama-quantize"):
    !./llama-quantize "{GGUF_F16}" "{GGUF_FINAL}" Q4_K_M
else:
    # Fallback to direct q8_0 conversion if quantize binary build is skipped
    !python3 convert_hf_to_gguf.py "{OUTPUT_DIR}" --outfile "{GGUF_FINAL}" --outtype q8_0

print(f"Conversion complete! Final 7B GGUF created at: {GGUF_FINAL}")

### 6. Download the 7B Model File
You can save the model directly to your Google Drive or download it to your local machine.

In [ ]:
from google.colab import drive
import shutil

try:
    drive.mount('/content/drive')
    shutil.copy(GGUF_FINAL, '/content/drive/MyDrive/iris-qwen2.5-coder-7b-heretic.Q4_K_M.gguf')
    print("Successfully saved 7B GGUF to your Google Drive under MyDrive!")
except Exception as e:
    print("Google Drive mount skipped. You can download via the file explorer on the left or colab files download:")
    from google.colab import files
    files.download(GGUF_FINAL)